# Solving SAT Problem Using Grover's Algorithm Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the "Solving SAT Problem Using Grover's Algorithm" kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import Qubits


## Problem 1. Evaluate AND operator

The goal is to flip the qubit $\ket{y}$ if and only if all qubits in the register $\ket{x}$ are in the state $\ket{1}$.

The required unitary $U_{and}$ is such that:
$$U_{and}\ket{x}\ket{y} = \begin{cases} 
          \ket{x}\ket{y} & \text{if }x \neq 1...1 \\
          \ket{x}X\ket{y} & \text{if }x = 1...1 
       \end{cases}$$

This transformation can be implemented as a controlled X gate, with the input register $\ket{x}$ as the control and the target qubit $\ket{y}$ as the target.

In [ ]:
def oracle_and(x: Qubits, y: Qubits) -> None:
    y.x(cond=x)

## Problem 2. Evaluate OR operator

The goal is to flip the qubit $\ket{y}$ if and only if at least one of the qubits in the register $\ket{x}$ is in the state $\ket{1}$.

The required unitary $U_{or}$ is such that:
$$U_{or}\ket{x}\ket{y} = \begin{cases} 
          \ket{x}\ket{y} & \text{if }x = 0...0 \\
          \ket{x}X\ket{y} & \text{if }x \neq 0...0
       \end{cases}$$

This transformation can be implemented as a sequence of two steps:

1. Flip the state of the target qubit if $x = 0...0$ using a controlled X gate with condition `x == 0`.
2. Flip the state of the target qubit using an $X$ gate unconditionally. This will negate the results of the previous step for $x = 0...0$,
   making sure that overall the state of the target qubit is flipped if $x \neq 0...0$.

In [ ]:
def oracle_or(x: Qubits, y: Qubits) -> None:
    y.x(cond=x == 0)
    y.x()

## Problem 3. Evaluate one clause of a SAT formula

This task involves evaluating a clause which is a disjunction (OR) of negated and non-negated variables encoded in the input $x$. 
This can be done in two steps:

1. First, flip the qubits which are negated in `literals` (and later undo this operation).  
   To do this, apply an $X$ gate to qubit $j$ if and only if the clause has a term of the form `(j, False)`.
2. Use the $U_{or}$ unitary (implemented by the operation `oracle_or`) to calculate the clause.  
   To do this, you need to first construct an array `controls` - all qubits which are included as a negated or non-negated variable in the clause. Then, you can apply the `oracle_or` operation to qubits `controls` and `y`.

Alternatively, you can think clause evaluation in terms of gates controlled on patterns. We've seen earlier that an OR operation can be represented as a pair of gates - an X gate and a controlled X gate with control pattern "all qubits are $0$". To express this for the clause, we can use Workbench feature that allows us to store the information about the control pattern in the control register itself - the `cond_xor` mask (see the [tutorial on controlled gates](https://docs.construct.psiquantum.com/workbench/new-tutorials/Controlled-Gates.html#modifying-cond_xor-masks) for details).

We will construct the Qubits object `controls` describing both the list of qubits in the clause and the corresponding pattern. To do this, we'll iterate through the list of literals `ind, pos` and append either the corresponding single-qubit Qubits object `x[ind]` or its negation (the same qubit with the bitmask set to $1$) `~x[ind]` to the `controls` register using the `|` operator. For simplicity, we can start with the Qubit object that corresponds to the first literal instead of an empty Qubits object.

Then, we'll pass this object to the `oracle_or` as the input register.

Note that we will need to uncompute the effect of `oracle_sat_clause` in the later tasks. We could implement it as a Qubrick to automate that, or we can simply notice that this computation is self-adjoint and call `oracle_sat_clause` itself whenever we need to uncompute.

In [ ]:
def oracle_sat_clause(x: Qubits, y: Qubits, literals: list[tuple[int, bool]]) -> None:
    # Build a mask of control qubits for this clause
    ind, pos = literals[0]
    controls = x[ind] if pos else ~x[ind]
    for ind, pos in literals[1:]:
        controls |= x[ind] if pos else ~x[ind]

    # Calculate OR of the literals in the clause
    oracle_or(controls, y)

## Problem 4. Evaluate SAT formula

This problem consists of evaluating a conjunction (AND) of results of multiple clause evaluations. Each clause individually can be evaluated using the code you've written in the previous problem. The computation results of these clauses must be stored temporarily in freshly allocated qubits. Then the conjunction of these results can be computed using `oracle_and` from the first problem.

Let's denote the number of clauses in the formula as $m$. The steps for implementing the SAT oracle will be:

1. Allocate an array of $m$ qubits `clause_ok` in the state $\ket{0}$.
2. Evaluate each clause using `oracle_sat_clause` from the previous problem, with the corresponding element of `clause_ok` as the target qubit.
3. Evaluate the SAT formula using `oracle_and` implemented in the first problem with `clause_ok` as the input register and `y` as the target qubit.
4. Undo step 2 (by repeating clause evaluation) to restore the auxiliary qubits back into the $\ket{0}$ state before releasing them.
5. Release `clause_ok` qubits explicitly.

In [ ]:
def oracle_sat_formula(x: Qubits, y: Qubits, clauses: list[list[tuple[int, bool]]]) -> None:
    num_clauses = len(clauses)

    # Allocate auxiliary qubits for clause evaluation
    clause_ok = Qubits(num_clauses, "clause_ok", x.qpu)

    # Evaluate clauses and store results in corresponding auxiliary qubits
    for ind in range(num_clauses):
        oracle_sat_clause(x, clause_ok[ind], clauses[ind])

    # Evaluate the formula as the AND of auxiliary qubits
    oracle_and(clause_ok, y)

    # Uncompute clauses evaluation by repeating it
    for ind in range(num_clauses):
        oracle_sat_clause(x, clause_ok[ind], clauses[ind])

    # Release qubits
    clause_ok.release()

## Problem 5. Evaluate "Exactly one 1" operator

Consider the set of all bit strings $x$ of length $n$ which have only one bit of $x$ equal to $1$.   
This set of bit strings is $S=\{00...01, 00...10, ..., 00..1..00, ..., 01...00, 10...00\}$, 
or, if we convert the bit strings to integers, 
$$S=\{1,2,4,..2^i,..,2^{n-1}\} = \{2^k: 0 \le k \le n-1\}$$

You need to implement an oracle that flips $\ket{y}$ if the input basis state $\ket{x}$ corresponds to one of the bit strings in $S$.
The easiest way to do this is to use $n$ controlled $X$ gates, with the qubit `y` flipped if and only if the control register `x` is in a particular state of the form $2^k$.

> If you needed to solve larger instances of this problem, you could've used a more elaborate algorithm, for example, count the number of $1$ bits in the input register and flip the qubit `y` if this number is $1$. But since our problem instance has only three qubits, it's easier to just use three gates with three qubits as controls.

In [ ]:
def oracle_exactly1one(x: Qubits, y: Qubits) -> None:
    for ind in range(x.num_qubits):
        y.x(cond=x == (2 ** ind))

## Problem 6. Evaluate one clause of exactly-1 3-SAT formula

This problem is very similar to the problem of evaluating a clause of a regular SAT formula: you need to evaluate a certain condition (exactly-one-1 instead of OR) on a set of literals defined in the same way. This means that you can borrow the logic of extracting the literals into a register of control qubits from problem 3 and just change the last step of the solution, calling `oracle_exactly1one` instead of `oracle_or`.

In [ ]:
def oracle_exactly1one_sat_clause(x: Qubits, y: Qubits, literals: list[tuple[int, bool]]) -> None:
    # Build a mask of control qubits for this clause
    ind, pos = literals[0]
    controls = x[ind] if pos else ~x[ind]
    for ind, pos in literals[1:]:
        controls |= x[ind] if pos else ~x[ind]

    # Calculate exactly-one-one logic of the literals in the clause
    oracle_exactly1one(controls, y)    

## Problem 7. Evaluate exactly-1 3-SAT formula

This problem, again, is very similar to the problem of evaluating a regular SAT formula (problem 4): it consists of a conjunction (AND) of results of multiple clause evaluations. This time, though, each clause has to be evaluated using the exactly-one-1 policy, which is the problem you solved in the previous exercise.

With this replacement, the code that evaluates the whole formula will be very similar to that from problem 4.

In [ ]:
def oracle_exactly1one_sat_formula(x: Qubits, y: Qubits, clauses: list[list[tuple[int, bool]]]) -> None:
    num_clauses = len(clauses)

    # Allocate auxiliary qubits for clause evaluation
    clause_ok = Qubits(num_clauses, "clause_ok", x.qpu)

    # Evaluate clauses and store results in corresponding auxiliary qubits
    for ind in range(num_clauses):
        oracle_exactly1one_sat_clause(x, clause_ok[ind], clauses[ind])

    # Evaluate the formula as the AND of auxiliary qubits
    oracle_and(clause_ok, y)

    # Uncompute clauses evaluation by repeating it
    for ind in range(num_clauses):
        oracle_exactly1one_sat_clause(x, clause_ok[ind], clauses[ind])

    # Release qubits
    clause_ok.release()

> Copyright (c) 2026 PsiQuantum